Control N°1 Optimización


Importacion de librerias y seleccion del Solver

In [1]:
%pip install -q amplpy
from amplpy import AMPL, ampl_notebook



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
ampl = ampl_notebook(modules=['highs'], license_uuid = "default")

AMPL Version 20250901 (MSVC 19.44.35215.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "C:\Users\Manguera\AppData\Roaming\Python\Python313\site-packages\ampl_module_base\bin\ampl.lic".



## **Problema N°1: Optimización de Procesos de Producción**

El objetivo es determinar las horas de operación de dos procesos distintos para minimizar el costo total diario, asegurando que se cumplan las demandas mínimas de tres químicos específicos.

### **Variables de Decisión**
* $x_1$: Horas de operación del **Proceso 1** por día.
* $x_2$: Horas de operación del **Proceso 2** por día.

### **Función Objetivo**
Minimizar el costo total operativo diario ($Z$):
$$\text{Min } Z = 40x_1 + 10x_2$$

### **Restricciones**
Sujeto a:

1. **R1** (Demanda del químico A): $3x_1 + x_2 \geq 40$
2. **R2** (Demanda del químico B): $x_1 + x_2 \geq 15$
3. **R3** (Demanda del químico C): $x_1 \geq 5$
4. **R4** (No negatividad): $x_1, x_2 \geq 0$

In [ ]:
# Definir el modelo AMPL
model ="""
var x1 >= 0;  # Horas del proceso 1
var x2 >= 0;  # Horas del proceso 2

# Función objetivo:
minimize costo: 40*x1 + 10*x2;

# Restricciones
subject to
  demanda_A: 3*x1 + x2 >= 40;    # Demanda del químico A
  demanda_B: x1 + x2 >= 15;      # Demanda del químico B
  demanda_C: x1 >= 5;             # Demanda del químico C
"""

# Resolver el modelo

ampl.reset()
ampl.eval(model)
ampl.option["solver"] = "highs"
ampl.solve()

# Mostrar resultados
print("Estado solución:", ampl.solve_result)
print(f"\nValor óptimo: {ampl.obj['costo'].value():.2f}")
print(f"\nVariables de decisión:")
print(f"x1 (Proceso 1): {ampl.var['x1'].value():.2f} horas")
print(f"x2 (Proceso 2): {ampl.var['x2'].value():.2f} horas")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 450
0 simplex iterations
0 barrier iterations
Estado solución: solved

Valor óptimo (Z): 450.00

Variables de decisión:
x1 (Proceso 1): 5.00 horas
x2 (Proceso 2): 25.00 horas


## Modelo Matemático: Mezcla y Procesamiento con Subproductos

Este es un problema de Programación Lineal (PL) de mezcla y procesamiento con subproductos (desechos) que deben ser balanceados o gestionados.

### Variables de Decisión
* $M_1$: Libras de materia prima compradas para elaborar el Insumo 1.
* $M_2$: Libras de materia prima compradas para elaborar el Insumo 2.
* $A$: Onzas de Producto A producidas (procesos tipo 1).
* $B$: Onzas de Producto B producidas (procesos tipo 2).
* $C$: Onzas de Producto C fabricadas.
* $D$: Onzas de Producto D fabricadas.
* $R$: Onzas de desecho líquido vertidas al río.

### Función Objetivo
Maximizar la utilidad ($Z$), que es el ingreso por ventas menos los costos totales (costo de compra y procesamiento de materia prima, más costos de los procesos de los productos):
$$\text{Max } Z = 17A + 16B + 7C + 2D - 8M_1 - 10M_2$$

### Restricciones
Sujeto a:
$$ 
\begin{align*}
2M_1 &= 2A + B + 2C && \text{(Balance de Insumo 1)} \\
3M_2 &= A + 2B + 2D && \text{(Balance de Insumo 2)} \\
A + 0.8B &= R + 0.8C + 1.2D && \text{(Balance de Desecho Líquido)} \\
R &\leq 1000 && \text{(Límite Gubernamental del Río)} \\
A &\leq 5000 && \text{(Demanda Máxima Producto A)} \\
B &\leq 5000 && \text{(Demanda Máxima Producto B)} \\
2M_1 + 2M_2 + 2A + 3B + C + D &\leq 6000 && \text{(Disponibilidad de Tiempo Total)} \\
M_1, M_2, A, B, C, D, R &\geq 0 && \text{(No negatividad)}
\end{align*}
$$

In [7]:
# Definir el modelo AMPL para el segundo problema
model2 = """
var M1 >= 0;  # Libras de materia prima para Insumo 1
var M2 >= 0;  # Libras de materia prima para Insumo 2
var A >= 0;   # Onzas de Producto A
var B >= 0;   # Onzas de Producto B
var C >= 0;   # Onzas de Producto C
var D >= 0;   # Onzas de Producto D
var R >= 0;   # Onzas de desecho líquido

# Función objetivo: Maximizar utilidad
maximize utilidad: 17*A + 16*B + 7*C + 2*D - 8*M1 - 10*M2;

# Restricciones
subject to
  balance_insumo1: 2*M1 = 2*A + B + 2*C;          # Balance de Insumo 1
  balance_insumo2: 3*M2 = A + 2*B + 2*D;          # Balance de Insumo 2
  balance_desecho: A + 0.8*B = R + 0.8*C + 1.2*D; # Balance de Desecho Líquido
  limite_rio: R <= 1000;                           # Límite Gubernamental del Río
  demanda_A: A <= 5000;                            # Demanda Máxima Producto A
  demanda_B: B <= 5000;                            # Demanda Máxima Producto B
  tiempo_total: 2*M1 + 2*M2 + 2*A + 3*B + C + D <= 6000;  # Disponibilidad de Tiempo Total
"""

# Resolver el modelo
ampl.reset()
ampl.eval(model2)
ampl.option["solver"] = "highs"
ampl.solve()

# Mostrar resultados
print("Estado solución:", ampl.solve_result)
print(f"\nValor óptimo de la utilidad: {ampl.obj['utilidad'].value():.2f}")
print(f"\nVariables de decisión:")
print(f"M1 (Materia prima Insumo 1): {ampl.var['M1'].value():.2f} libras")
print(f"M2 (Materia prima Insumo 2): {ampl.var['M2'].value():.2f} libras")
print(f"A (Producto A): {ampl.var['A'].value():.2f} onzas")
print(f"B (Producto B): {ampl.var['B'].value():.2f} onzas")
print(f"C (Producto C): {ampl.var['C'].value():.2f} onzas")
print(f"D (Producto D): {ampl.var['D'].value():.2f} onzas")
print(f"R (Desecho líquido): {ampl.var['R'].value():.2f} onzas")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 6366.336634
2 simplex iterations
0 barrier iterations
Estado solución: solved

Valor óptimo de la utilidad: 6366.34

Variables de decisión:
M1 (Materia prima Insumo 1): 1356.44 libras
M2 (Materia prima Insumo 2): 386.14 libras
A (Producto A): 1158.42 onzas
B (Producto B): 0.00 onzas
C (Producto C): 198.02 onzas
D (Producto D): 0.00 onzas
R (Desecho líquido): 1000.00 onzas


## Problema 3: Refinería Melrose

Este es un modelo complejo de mezcla (blending) donde los subproductos obtenidos pueden ser transformados mediante un proceso de mejora (desintegración catalítica).

### Variables de Decisión
* $BC_1, BC_2, BC_3$: Barriles de **C**rudo procesados mediante los métodos 1, 2 y 3.
* $M_{6 \to 8}, M_{8 \to 10}$: Barriles de crudo sometidos a **M**ejora para subir de grado.
* $Gas_6, Gas_8, Gas_{10}$: Barriles de grado 6, 8 y 10 destinados a la mezcla de **Gas**olina final.
* $Ace_6, Ace_8, Ace_{10}$: Barriles de grado 6, 8 y 10 destinados a la mezcla de **Ace**ite final.
* $BD_6, BD_8, BD_{10}$: Barriles de grado 6, 8 y 10 **D**esechados.

### Función Objetivo
Maximizar la utilidad ($Z$), calculada como los ingresos por venta de gasolina (\$8/barril) y aceite (\$6/barril), menos los costos operativos (procesamiento, mejora y desecho).

*(Nota: Como no se entregaron los coeficientes exactos de costos en el enunciado, se expresan de manera conceptual en la función).*
$$\text{Max } Z = 8(Gas_6 + Gas_8 + Gas_{10}) + 6(Ace_6 + Ace_8 + Ace_{10}) - \text{Costos de Procesos, Mejora y Desecho}$$

### Restricciones
Sujeto a:
$$ 
\begin{align*}
\text{Salida Grado } i &= Gas_i + Ace_i + M_{i \to j} + BD_i && \text{(Equilibrio de salidas hacia Venta, Mejora y Desecho)} \\
-3Gas_6 - Gas_8 + Gas_{10} &\geq 0 && \text{(Grado Gasolina } \geq 9 \text{, linealizada)} \\
-Ace_6 + Ace_8 + 3Ace_{10} &\geq 0 && \text{(Grado Aceite } \geq 7 \text{, linealizada)} \\
BC_i, M_{i \to j}, Gas_i, Ace_i, BD_i &\geq 0 && \text{(No negatividad)}
\end{align*}
$$

El cálculo original del grado promedio ponderado para la gasolina es una fracción: 
$\frac{6Gas_6 + 8Gas_8 + 10Gas_{10}}{Gas_6 + Gas_8 + Gas_{10}} \geq 9$


In [8]:
model_melrose = """
# Variables de Crudo
var BC1 >= 0; var BC2 >= 0; var BC3 >= 0;

# Variables de Mejora
var M6_8 >= 0; var M8_10 >= 0;

# Variables de Gasolina y Aceite
var Gas6 >= 0; var Gas8 >= 0; var Gas10 >= 0;
var Ace6 >= 0; var Ace8 >= 0; var Ace10 >= 0;

# Variables de Desecho
var BD6 >= 0; var BD8 >= 0; var BD10 >= 0;

maximize utilidad: 
    8*(Gas6 + Gas8 + Gas10) + 6*(Ace6 + Ace8 + Ace10) 
    - (3.4*BC1 + 3.0*BC2 + 2.6*BC3)
    - 1.3*M6_8 - 2.0*M8_10 
    - 0.2*(BD6 + BD8 + BD10);

s.t.
# Equilibrio de salidas hacia Venta, Mejora y Desecho
bal_6:  0.2*BC1 + 0.3*BC2 + 0.4*BC3 = Gas6 + Ace6 + M6_8 + BD6;
bal_8:  0.2*BC1 + 0.3*BC2 + 0.4*BC3 + M6_8 = Gas8 + Ace8 + M8_10 + BD8;
bal_10: 0.6*BC1 + 0.4*BC2 + 0.2*BC3 + M8_10 = Gas10 + Ace10 + BD10;

# Restricciones de demanda máxima
max_gasolina: Gas6 + Gas8 + Gas10 <= 2000;
max_aceite: Ace6 + Ace8 + Ace10 <= 600;

# Calidad: el peso ponderado debe ser mayor o igual al target (9 para gas, 7 para aceite)
calidad_gasolina: -3*Gas6 - 1*Gas8 + 1*Gas10 >= 0; 
calidad_aceite:   -1*Ace6 + 1*Ace8 + 3*Ace10 >= 0;
"""

# Ejecución en la API de AMPL
ampl.reset()
ampl.eval(model_melrose)
ampl.option["solver"] = "highs"
ampl.solve()

# Extracción de resultados usando las nuevas variables (respetando mayúsculas)
gasolina = sum([ampl.get_value(f'Gas{i}') for i in [6,8,10]])
aceite = sum([ampl.get_value(f'Ace{i}') for i in [6,8,10]])

print(f"Barriles Gasolina Final: {gasolina}")
print(f"Barriles Aceite Final: {aceite}")
print(f"Utilidad Maximizada: USD {ampl.get_value('utilidad')}")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 11230
9 simplex iterations
0 barrier iterations
Barriles Gasolina Final: 2000
Barriles Aceite Final: 600
Utilidad Maximizada: USD 11230
